# 9A · The Anatomy of a Time Series — Decomposition
### Financial Analytics — Module 9 · Lab 1

Every time series is a sentence built from four parts:

> **observed = trend + seasonality + noise** (± the occasional structural break)

**Decomposition** is reading the sentence — separating the parts so each can be understood, modelled, and forecast on its own terms. Today you'll do something unusually honest: **build a series from known parts, then test whether decomposition can recover them.** When ground truth is known, you learn what the tool actually does — and where it smears.

> ### 🛡️ Bias Check (the Part D ritual — fill before running anything)
> **Look-ahead:** every statistic below uses only data available *at that point in time*? ☐
> **Survivorship:** who/what is missing from this series, and why? ☐
> **Point-in-time:** original prints, or restated history? ☐
> **Regime:** any structural breaks inside the window? ☐
>
> *For this lab: the sales series is synthetic (fully documented, no survivorship); NIFTY carries its known regime shift and two missing days from the registry. Note them; proceed.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(9)     # fixed seed: fully reproducible (auditability pillar)

---
## 1. Build the series — MoneyMart's monthly sales

MoneyMart's annual P&L hides a monthly rhythm every Indian retailer knows: **the festive spike**. October–November (Navratri→Diwali) sales run far above trend; the months after sag. We construct 6 years of monthly sales with three explicit components:

In [ ]:
months = pd.date_range("2020-01-31", periods=72, freq="ME")

# COMPONENT 1 - trend: steady growth in Rs crore/month
true_trend = 520 + 4.2*np.arange(72)

# COMPONENT 2 - seasonality: multiplicative festive pattern (Oct-Nov spike, Dec hangover)
seasonal_index = {1:0.94, 2:0.92, 3:0.98, 4:1.00, 5:0.97, 6:0.95,
                  7:0.98, 8:1.02, 9:1.05, 10:1.22, 11:1.14, 12:0.83}
true_seasonal = np.array([seasonal_index[m.month] for m in months])

# COMPONENT 3 - noise: the ordinary chaos of commerce
noise = rng.normal(1.0, 0.03, 72)

sales = pd.Series(true_trend * true_seasonal * noise, index=months, name="sales_cr")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(sales, color="#2563EB", lw=1.4)
for y in range(2020, 2026):
    ax.axvspan(pd.Timestamp(f"{y}-10-01"), pd.Timestamp(f"{y}-11-30"), color="#F59E0B", alpha=0.12)
ax.set_title("MoneyMart monthly sales — festive season shaded. Can you SEE the three components?",
             loc="left", fontweight="bold")
ax.set_ylabel("Rs crore"); plt.tight_layout(); plt.show()

Note the model choice: seasonality here **multiplies** the trend (a 22% October lift on a bigger base is more rupees each year). The alternative is **additive** (a fixed +₹X every October). Eyeball test: if the seasonal swings **grow with the level**, think multiplicative; if they stay constant-sized, additive. Choosing wrong doesn't crash anything — it quietly mis-sizes every seasonal forecast.

---
## 2. Recover the trend — the centred moving average

A 12-month moving average steamrolls the seasonality flat (every window contains exactly one full cycle, so the highs and lows cancel), leaving the trend:

In [ ]:
# Centred 12-month MA: average of a window balanced around each point
trend_est = sales.rolling(12, center=True).mean()

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(sales, color="#94A3B8", lw=0.9, label="observed")
ax.plot(trend_est, color="#DC2626", lw=2, label="12-mo centred MA (trend estimate)")
ax.plot(pd.Series(true_trend, index=months), color="#16A34A", lw=1.4, ls="--", label="TRUE trend (we built it!)")
ax.legend(); ax.set_title("Recovered vs true trend — close, with fuzzy edges", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

**Two honest observations.** The recovered trend tracks the truth well — and it's *missing at both ends* (a centred window needs 6 months on each side, so the first and last half-years are blank). Edge-blindness is the price of centring, and it matters: the most recent trend — the part you most want — is exactly what this estimator can't see. (Forecasting methods in 9C handle the edge differently.)

---
## 3. Recover the seasonality — the seasonal index

In [ ]:
# Detrend (divide, because our world is multiplicative), then average by calendar month
detrended = sales / trend_est
est_index = detrended.groupby(detrended.index.month).mean()
est_index = est_index / est_index.mean()          # normalise so the indices average to 1.0

comp = pd.DataFrame({"estimated": est_index.round(3),
                     "true": pd.Series(seasonal_index).round(3)})
print(comp.to_string())
print(f"\nMax recovery error: {(comp.estimated-comp.true).abs().max():.3f}  <- decomposition WORKS")

In [ ]:
# The seasonal fingerprint - a chart every retail/banking analyst keeps pinned up
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(range(1,13), est_index.values, color=["#F59E0B" if m in (10,11) else "#2563EB" for m in range(1,13)])
ax.axhline(1.0, color="black", lw=0.8)
ax.set_xticks(range(1,13), ["J","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_title("Seasonal index: October runs ~22% above trend, December ~17% below", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

Read the index as a **translator**: October's 1.22 means "an October ₹800 cr is really a trend-level ₹656 cr wearing festive clothes." This is how professionals answer the question that wrecks naive dashboards every December: *"sales fell 30% from November — panic?"* No: **deseasonalise first, then judge.**

## 4. The residual — what's left must be boring

In [ ]:
residual = sales / (trend_est * pd.Series([seasonal_index[m.month] for m in months], index=months))
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.plot(residual, color="#7C3AED", lw=0.9)
ax.axhline(1.0, color="black", lw=0.8)
ax.set_title("Residual after removing trend & season: structureless wobble = decomposition captured the signal",
             loc="left", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Residual std: {residual.std():.3f}  (we injected 0.03 - recovered almost exactly)")

**The residual is the diagnostic.** If it's boring — structureless wobble around 1.0 — the decomposition captured the signal. If it still shows waves or trends, your components missed something, and modelling should not proceed. *(Sound familiar? This is Module 6's residual-as-lie-detector, promoted to time series.)*

---
## 5. Now the humbling one: decompose NIFTY

In [ ]:
import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
px = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"]).set_index("date")
m_close = px["close"].resample("ME").last().dropna()

mtrend = m_close.rolling(12, center=True).mean()
mseason = (m_close / mtrend).groupby(lambda d: d.month).mean()
print("NIFTY 'seasonal index' by month:")
print(mseason.round(3).to_string())
print(f"\nSpread: {mseason.min():.3f} to {mseason.max():.3f} - a whisper vs sales' 0.83-1.22 roar.")

**The honest lesson of the whole notebook:** MoneyMart's sales have loud, exploitable structure. NIFTY's "seasonality" is a whisper indistinguishable from noise — and Module 6 told you why: *any loud calendar pattern in prices would be traded away by whoever saw it first.* Decomposition is a superb tool **for series where structure is allowed to survive** — sales, deposits, loan demand, power consumption. Applying it to asset prices mostly decomposes noise into prettier noise.

### ✏️ Exercises
1. **Additive vs multiplicative, felt:** rebuild the sales series with *additive* seasonality (`true_trend + 60*np.array([...centered indices...])`). Decompose it multiplicatively anyway. Where does the recovered index go wrong, and in which years is it worst?
2. **The December defence:** sales drop from ₹905 cr (Nov) to ₹640 cr (Dec). Using the seasonal indices, compute the *deseasonalised* levels of both months. Did the business actually weaken?
3. **Break the tool:** inject a level shift (+₹120 cr from month 40 onward — a new store format). Re-run the decomposition. Where does the break's damage show up — trend, season, or residual? What does that tell you to check *before* trusting any decomposition?

---
*AI disclosure: ______*

In [ ]:
# workspace
